Check this tutorial on CNN 

http://learnopencv.com/understanding-convolutional-neural-networks-cnn/

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
from torchvision import transforms
from torch.utils.tensorboard import SummaryWriter
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
import numpy as np
import os

In [2]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print(device)
writer = SummaryWriter("runs/cifar10_experiment")
transform = transforms.Compose(
    [transforms.ToTensor(), transforms.Normalize((0.5,), (0.5,))]
)
data_train = torchvision.datasets.CIFAR10(root='./data', train=True, download=True, transform=transform)
dataloader_train = DataLoader(data_train, batch_size=64, shuffle=True)
data_test = torchvision.datasets.CIFAR10(root='./data', train=False, download=True, transform=transform)
dataloader_test = DataLoader(data_test, batch_size=64, shuffle=False)



cuda


In [3]:
class MyCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(in_channels=3, out_channels=16, kernel_size=3, padding=1)
        self.pool1 = nn.MaxPool2d(2, 2)
        self.conv2 = nn.Conv2d(16, 32, 3, padding=1)
        self.fc1 = nn.Linear(32 * 8 * 8, 128)
        self.fc2 = nn.Linear(128, 10)

    def forward(self, x):
        self.activation_map = F.relu(self.conv1(x))
        x = self.pool1(self.activation_map)
        x = F.relu(self.conv2(x))
        x = self.pool1(x)
        x = x.view(-1, 32 * 8 * 8)
        x = F.relu(self.fc1(x))
        x = self.fc2(x)
        return x

In [12]:
model = MyCNN().to(device)
criterion  = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(),lr=0.001)

In [5]:
def plot_activation_map(activation_tensor, writer, step):
    # Only first 8 activations from the first image in batch
    act = activation_tensor[0, :8, :, :].detach().cpu()
    grid = torchvision.utils.make_grid(act.unsqueeze(1), normalize=True, scale_each=True)
    writer.add_image("Activations/Conv1", grid, step)

In [13]:
from tqdm.notebook import tqdm
for epoch in range(5):
    running_loss = 0.0
    total, correct = 0.0,0.0

    plot_now = True
    for images,labels in tqdm(dataloader_train):
        images = images.to(device)
        labels=labels.to(device)
        optimizer.zero_grad()
        model.train()
        pred = model(images)

        loss = criterion(pred, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
        
        _, predicted = torch.max(pred.data,1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()
        if plot_now:
            plot_activation_map(model.activation_map, writer, epoch)
    epoch_loss = running_loss / len(dataloader_train)
    epoch_acc = correct / total
    writer.add_scalar("Loss/train", epoch_loss, epoch)
    writer.add_scalar("Accuracy/train", epoch_acc, epoch)




  0%|          | 0/782 [00:00<?, ?it/s]

  0%|          | 0/782 [00:00<?, ?it/s]

  0%|          | 0/782 [00:00<?, ?it/s]

  0%|          | 0/782 [00:00<?, ?it/s]

  0%|          | 0/782 [00:00<?, ?it/s]

In [14]:
# preict
correct = 0
total = 0
with torch.no_grad():
    model.eval()
    for images, labels in dataloader_test:
        images = images.to(device)
        labels=labels.to(device)
        preds = model(images)
        _, predicted = torch.max(preds,axis=1 )
        correct += (predicted == labels).sum().item()
        total += labels.size(0)

test_acc = 100 * correct / total
writer.add_scalar("Accuracy/test", test_acc)
print(f"Test Accuracy: {test_acc:.2f}%")      
writer.close()

Test Accuracy: 69.39%


Note: I got a poor accuracy (39%) by lr=0.01. 